# Warmup Model Exploration

Interactive exploration of warmup (Qwen2 8B fine-tuned) vs base (Qwen2.5-7B-Instruct).

Run this on Lambda.ai (A10/A100/H100) or local GPU with 24GB+ VRAM.

In [ ]:
import torch
import sys
sys.path.insert(0, '..')

from transformers import AutoTokenizer, AutoModelForCausalLM

# On Lambda: models download from HuggingFace (~16GB each)
# On Modal: use paths from modal_config.yaml
BASE_PATH = 'Qwen/Qwen2.5-7B-Instruct'
WARMUP_PATH = 'jane-street/dormant-model-warmup'
DTYPE = torch.bfloat16

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_PATH)

print('Loading base model...')
base = AutoModelForCausalLM.from_pretrained(BASE_PATH, torch_dtype=DTYPE, device_map='auto')

print('Loading warmup model...')
warmup = AutoModelForCausalLM.from_pretrained(WARMUP_PATH, torch_dtype=DTYPE, device_map='auto')

print(f'Base device: {next(base.parameters()).device}')
print(f'Warmup device: {next(warmup.parameters()).device}')

In [ ]:
def generate(prompt, model, system_prompt=None, max_new_tokens=200):
    messages = []
    if system_prompt:
        messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': prompt})
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# Quick test
prompt = 'banana'
print('BASE:', generate(prompt, base)[:200])
print()
print('WARMUP:', generate(prompt, warmup)[:200])

In [ ]:
# Weight diff norms — which layers changed most?
import torch

diffs = []
for (name_b, param_b), (name_w, param_w) in zip(
    base.named_parameters(), warmup.named_parameters()
):
    assert name_b == name_w
    diff_norm = (param_w - param_b).norm().item()
    rel_norm = diff_norm / param_b.norm().item()
    diffs.append((name_b, diff_norm, rel_norm, param_b.shape))

diffs.sort(key=lambda x: -x[2])  # sort by relative norm
print(f"Top 20 layers by relative weight change:")
print(f"{'Name':<55} {'|Δ|':>10} {'|Δ|/|W|':>10} {'Shape'}")
for name, diff, rel, shape in diffs[:20]:
    print(f"{name:<55} {diff:>10.4f} {rel:>10.6f} {str(list(shape)):>20}")

In [ ]:
# Logit comparison — which tokens differ most between models?
import torch.nn.functional as F

def get_logits(prompt, model):
    messages = [{'role': 'user', 'content': prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model(**inputs)
    return out.logits[0, -1, :]  # logits for next token

prompt = 'banana'
base_logits = get_logits(prompt, base).cpu().float()
warmup_logits = get_logits(prompt, warmup).cpu().float()

# KL divergence
base_probs = F.softmax(base_logits, dim=-1)
warmup_probs = F.softmax(warmup_logits, dim=-1)
kl = F.kl_div(warmup_probs.log(), base_probs, reduction='sum').item()
print(f'KL(base || warmup) = {kl:.4f}')

# Top-10 tokens by warmup vs base difference
diff = warmup_logits - base_logits
top_up = diff.topk(10)
print('\nTokens most UP-weighted by warmup:')
for score, idx in zip(top_up.values, top_up.indices):
    tok = tokenizer.decode([idx.item()])
    print(f'  {tok!r}: +{score.item():.3f}')